In [0]:
#Imports
from pyspark.sql import functions as F
from datetime import datetime
import uuid

In [0]:
# Table configuration

silver_table = (
    "workspace.nyc_taxi_aws.silver_taxi_trips_curated"
)

quality_table = (
    "workspace.nyc_taxi_aws.data_quality_aws"
)

pipeline_name = "nyc_taxi_lakehouse_v2"
task_name = "data_quality"

start_time = datetime.now()

run_id = str(uuid.uuid4())

print(f"Data Quality Run ID: {run_id}")

Data Quality Run ID: eee36fdd-9eb9-402f-9b66-ccc649bc91b8


In [0]:
# Read Silver table

silver_df = spark.table(silver_table)

total_records = silver_df.count()

print(
    f"Total records: {total_records}"
)

Total records: 3839282


In [0]:
# Quality check function

def run_quality_check(
    check_name,
    table_name,
    total_records,
    failed_records,
    threshold_percentage=1.0
):

    failure_percentage = (
        failed_records / total_records * 100
        if total_records > 0
        else 100
    )

    status = (
        "PASS"
        if failure_percentage <= threshold_percentage
        else "FAIL"
    )

    return {
        "check_name": check_name,
        "table_name": table_name,
        "total_records": total_records,
        "failed_records": failed_records,
        "failure_percentage": failure_percentage,
        "status": status
    }

In [0]:
# Check 1: Trip distance must be positive

failed_trip_distance = (
    silver_df
    .filter(
        (F.col("trip_distance").isNull()) |
        (F.col("trip_distance") <= 0)
    )
    .count()
)

trip_distance_check = run_quality_check(
    check_name="trip_distance_positive",
    table_name=silver_table,
    total_records=total_records,
    failed_records=failed_trip_distance
)

In [0]:
# Check 2: Fare amount must be valid

failed_fare_amount = (
    silver_df
    .filter(
        (F.col("fare_amount").isNull()) |
        (F.col("fare_amount") < 0)
    )
    .count()
)

fare_amount_check = run_quality_check(
    check_name="fare_amount_valid",
    table_name=silver_table,
    total_records=total_records,
    failed_records=failed_fare_amount
)

In [0]:
# Check 3: Pickup timestamp must not be null

failed_pickup_time = (
    silver_df
    .filter(
        F.col("pickup_datetime").isNull()
    )
    .count()
)

pickup_time_check = run_quality_check(
    check_name="pickup_datetime_not_null",
    table_name=silver_table,
    total_records=total_records,
    failed_records=failed_pickup_time
)

In [0]:
# Check 4: Dropoff timestamp must not be null

failed_dropoff_time = (
    silver_df
    .filter(
        F.col("dropoff_datetime").isNull()
    )
    .count()
)

dropoff_time_check = run_quality_check(
    check_name="dropoff_datetime_not_null",
    table_name=silver_table,
    total_records=total_records,
    failed_records=failed_dropoff_time
)

In [0]:
# Check 5: Dropoff must occur after pickup

failed_timestamp_order = (
    silver_df
    .filter(
        F.col("dropoff_datetime")
        <= F.col("pickup_datetime")
    )
    .count()
)

timestamp_order_check = run_quality_check(
    check_name="dropoff_after_pickup",
    table_name=silver_table,
    total_records=total_records,
    failed_records=failed_timestamp_order
)

In [0]:
# Combine all quality results

quality_results = [
    trip_distance_check,
    fare_amount_check,
    pickup_time_check,
    dropoff_time_check,
    timestamp_order_check
]

quality_results_df = (
    spark.createDataFrame(quality_results)
    .withColumn(
        "execution_timestamp",
        F.current_timestamp()
    )
    .withColumn(
        "run_id",
        F.lit(run_id)
    )
)

display(quality_results_df)

check_name,failed_records,failure_percentage,status,table_name,total_records,execution_timestamp,run_id
trip_distance_positive,0,0.0,PASS,workspace.nyc_taxi_aws.silver_aws,3839282,2026-07-28T16:52:17.855Z,eee36fdd-9eb9-402f-9b66-ccc649bc91b8
fare_amount_valid,0,0.0,PASS,workspace.nyc_taxi_aws.silver_aws,3839282,2026-07-28T16:52:17.855Z,eee36fdd-9eb9-402f-9b66-ccc649bc91b8
pickup_datetime_not_null,0,0.0,PASS,workspace.nyc_taxi_aws.silver_aws,3839282,2026-07-28T16:52:17.855Z,eee36fdd-9eb9-402f-9b66-ccc649bc91b8
dropoff_datetime_not_null,0,0.0,PASS,workspace.nyc_taxi_aws.silver_aws,3839282,2026-07-28T16:52:17.855Z,eee36fdd-9eb9-402f-9b66-ccc649bc91b8
dropoff_after_pickup,0,0.0,PASS,workspace.nyc_taxi_aws.silver_aws,3839282,2026-07-28T16:52:17.855Z,eee36fdd-9eb9-402f-9b66-ccc649bc91b8


In [0]:
# Save quality results

(
    quality_results_df
    .select(
        "run_id",
        "check_name",
        "table_name",
        "failed_records",
        "total_records",
        "failure_percentage",
        "status",
        "execution_timestamp"
    )
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(quality_table)
)

In [0]:
# Calculate quality metrics

failed_records = (
    sum(
        result["failed_records"]
        for result in quality_results
    )
)

end_time = datetime.now()

duration_seconds = (
    end_time - start_time
).total_seconds()

pipeline_status = (
    "FAIL"
    if any(
        result["status"] == "FAIL"
        for result in quality_results
    )
    else "SUCCESS"
)

In [0]:
# Display final status

failed_checks = [
    result
    for result in quality_results
    if result["status"] == "FAIL"
]

if failed_checks:

    failed_check_names = [
        check["check_name"]
        for check in failed_checks
    ]

    raise Exception(
        f"Data quality checks failed: "
        f"{failed_check_names}"
    )

else:

    print(
        "All data quality checks passed successfully."
    )

All data quality checks passed successfully.
